# Sindhi-Jev: training (Kaggle GPU)

**Budget rule:** every session's GPU time is logged in `runs/<name>/gpu_time.json`. Main training = 25 GPU h total,
debugging = 5 h, ablations = 15 h, baselines = 8 h, reserve = 7 h (60 h overall). Do not go past a budget without asking.

Setup once: upload `dist/sindhijev_kaggle.zip` as a private dataset **sindhijev-code**. In this notebook:
Settings -> Accelerator **GPU T4 x1** (or P100), Internet **on**, Add data -> sindhijev-code.
To resume: also add the previous version of this notebook's output as data (it holds `runs/<name>/last.pt`).
Evaluation is done locally on CPU from the downloaded checkpoint (no GPU hours).

In [ ]:
import glob, os, shutil, subprocess, sys
# find the uploaded code wherever Kaggle mounted it (any dataset name / nesting)
hits = glob.glob('/kaggle/input/**/src/sindhijev/model.py', recursive=True)
if not hits:
    for root, dirs, files in os.walk('/kaggle/input'):
        print(root, dirs[:5], files[:5])
    raise SystemExit('sindhijev code not found: is the dataset added under Input (right panel)?')
SRC = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))  # .../src/sindhijev/model.py -> bundle root
print('code found at', SRC)
os.chdir(SRC)  # run in place (read-only input is fine: outputs go to /kaggle/working); copying 12k files took ~10 min
!pip -q install uharfbuzz==0.57.0 freetype-py==2.5.1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Settings
- `RUN`: run name (`main`, or an ablation name such as `abl_noscan`)
- `SET`: model overrides for ablations, e.g. `['use_scan=false']`
- `MAX_HOURS`: stop this session cleanly before Kaggle's 9 h limit
- `DEBUG = True` for the first session: 300 steps only, to measure speed and GPU use

In [ ]:
CONFIG = 'configs/model_v2.json'   # v1 model: configs/model_v1.json
RUN = 'main_v2'
SET = []            # ablations: ['use_scan=false'], ['use_render=false'], ['char_align=false'], ['char_head=false']
MAX_HOURS = 8.3
DEBUG = False
OUT = f'/kaggle/working/runs/{RUN}'
prev = glob.glob(f'/kaggle/input/**/runs/{RUN}/last.pt', recursive=True)
RESUME = ['--resume', prev[0]] if prev and not os.path.exists(f'{OUT}/last.pt') else []
print('resume from', RESUME)

In [ ]:
args = ['python', 'scripts/train.py', '--config', CONFIG, '--out', OUT,
        '--max-hours', str(MAX_HOURS), '--workers', '4'] + RESUME
if SET:
    args += ['--set'] + SET
if DEBUG:
    args += ['--epochs', '1', '--steps-per-epoch', '300', '--val-limit', '1000', '--budget-hours', '0.5']
print(' '.join(args))
subprocess.run(args, check=True)

## After the run
- GPU hours used by this run so far (report these to the project log):

In [ ]:
import json
g = json.load(open(f'{OUT}/gpu_time.json')) if os.path.exists(f'{OUT}/gpu_time.json') else {'sessions': []}
print(f"{sum(s['hours'] for s in g['sessions']):.2f} GPU h over {len(g['sessions'])} session(s)")
!tail -n 3 {OUT}/log.jsonl
# keep only checkpoints + logs in the output (Save Version -> 'Save & Run All' is NOT needed; use 'Quick Save')
shutil.copytree(OUT, f'/kaggle/working/runs_out/{RUN}', dirs_exist_ok=True)

## Speed check (DEBUG session)
In the training log, `s/step` with batch 128. If GPU utilisation (`nvidia-smi`) is well below 80% while the step time
is high, the CPU data generation is the bottleneck: generate shards in a **CPU** notebook with
`python scripts/make_shards.py --n 500000 --out /kaggle/working/shards --procs 4` (no GPU hours), attach that output,
and add `'--shards', *glob.glob('/kaggle/input/<shards>/shards/*.pkl')` to `args`.

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv